In [1]:
import scanpy as sc
import pandas as pd
import anndata as ad

In [ ]:
adata1 = sc.read_loom("/storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/data/merged_P1A_P1B.loom")

In [ ]:
adata2 = sc.read_loom("/storage/liuxiaodongLab/jiangjing/Projects/YutingFu/PD_YutingFu/scvelo/data/h9_all_months_harmony.loom")

In [ ]:
# 1. 删除以 mRatBN7.2_ 开头的基因
mask = ~adata2.var['var_names'].str.startswith("mRatBN7.2_")
adata2 = adata2[:, mask].copy()  # 使用 copy 避免碎片化

# ----------------------------
# 2. 清理剩余基因名的 GRCh38.109_ 前缀
# ----------------------------
# 方法：直接字符串替换（确保精确处理前缀）
adata2.var["var_names"] = adata2.var["var_names"].str.replace("GRCh38.109_", "", regex=False) 

In [ ]:
adata2.var = adata2.var.set_index("var_names")
adata2.obs = adata2.obs.set_index("obs_names")

# Step 2: 清理索引名称（可选）
adata2.var_names.name = None
adata2.obs_names.name = None

# Step 3: 处理重复项（如有需要）
adata2.var_names_make_unique()
adata2.obs_names_make_unique()

In [ ]:
# ======================
# 第一步：基因交集处理
# ======================
# 提取基因ID交集（需确保均为索引）
common_genes = adata1.var_names.intersection(adata2.var_names)
common_genes

In [ ]:
# 按交集过滤数据
adata1_filt = adata1[:, common_genes].copy()  # 保留所有obs，过滤var
adata2_filt = adata2[:, common_genes].copy()


In [ ]:
# ======================
# 第二步：obs元数据整合
# ======================
# 重命名批次列并过滤无关列
adata1_filt.obs = adata1_filt.obs[['batch']]  # 只保留batch列
adata2_filt.obs = adata2_filt.obs[['sample_id']].rename(columns={'sample_id': 'batch'})

In [ ]:
# ======================
# 第三步：对象合并
# ======================
# 使用指定参数合并（关键步骤）
merged_adata = ad.concat(
    [adata1_filt, adata2_filt],
    axis=0,  # 合并obs维度
    join="inner",  # 使用严格交集模式（确保结构一致性）
    label="data_source",  # 添加数据来源标识列
    keys=["adata1", "adata2"],  # 来源标识符
    merge="unique"  # 遇到冲突列时自动重命名
)

# ======================
# 第四步：layer数据对齐
# ======================
# 确保所有层存在（用零矩阵填充缺失层）
for layer in ['matrix', 'spliced', 'unspliced']:
    if layer not in merged_adata.layers:
        merged_adata.layers[layer] = csr_matrix(merged_adata.shape)

# ======================
# 第五步：索引去重
# ======================
merged_adata.obs_names_make_unique(join="-")
merged_adata.var_names_make_unique(join="-")

In [ ]:
# ======================
# 第六步：保存Loom文件
# ======================
merged_adata.write_loom(
    "merged.loom",
    write_obsm_varm=True  # 忽略非常用结构以提高兼容性
)
